# Generate COT Contract Mapping Summary CSV

Produces `docs/cot_contract_mapping_summary.csv`  -  a single flat file that maps every
COT contract we've identified to a future (CL, CO, HO, XB, QS), classifies it as
flat price or not, and records the rationale.

The mapping is defined as Python dicts below. Each future has:
- A set of **flat price codes** (selected)
- A set of **excluded codes** with category and rationale
- ICE Europe contracts (Brent, Gasoil) handled separately since they have no CFTC code

In [1]:
import pandas as pd
import csv
from pathlib import Path

## Configuration  -  Flat Price Codes Per Future

Each entry: `code: (contract_type, category, rationale)`

In [2]:
FLAT_PRICE = {
    "WTI Crude Oil": {
        "bloomberg_ticker": "CL",
        "codes": {
            "067651": ("Physically-delivered futures", "Flat price - benchmark",
                       "The benchmark WTI contract. Bloomberg CL maps to this."),
            "067411": ("Futures (WTI-referenced)", "Flat price - benchmark",
                       "ICE-listed WTI lookalike, reported by CFTC. Aggregated into CL by Bloomberg."),
            "06765I": ("Cash-settled futures", "Flat price - financial",
                       "WTI financial settlement - flat price exposure to WTI."),
            "067A55": ("Cash-settled micro futures", "Flat price - financial",
                       "Micro-sized WTI flat price contract."),
            "067655": ("E-mini futures", "Flat price - financial",
                       "Smaller-sized WTI flat price contract."),
            "06765C": ("Asian-style options", "Flat price - options",
                       "Options on WTI flat price."),
            "06765B": ("European-style options", "Flat price - options",
                       "Options on WTI flat price."),
            "06741Q": ("Cash-settled futures", "Flat price - financial",
                       "ICE Energy Div WTI flat price."),
            "06765A": ("Calendar swap", "Flat price - calendar swap",
                       "WTI calendar swap - settles against average price. Added per Marouen."),
            "06765G": ("Calendar swap", "Flat price - calendar swap",
                       "Dubai crude calendar swap. Added per Marouen."),
        },
    },
    "Brent Crude Oil": {
        "bloomberg_ticker": "CO",
        "codes": {
            "06765J": ("Cash-settled futures", "Flat price - financial",
                       "Main NYMEX Brent lookalike - cash-settled."),
            "06765T": ("Physically-delivered futures", "Flat price - benchmark",
                       "Last trading day settlement."),
            "06765Y": ("Asian-style options", "Flat price - options",
                       "Options on Brent flat price."),
            "06765X": ("European-style options", "Flat price - options",
                       "Options on Brent flat price."),
            "06765N": ("Calendar swap", "Flat price - calendar swap",
                       "Brent (ICE) calendar swap. Added per Marouen."),
        },
    },
    "Heating Oil / ULSD": {
        "bloomberg_ticker": "HO",
        "codes": {
            "022651": ("Physically-delivered futures", "Flat price - benchmark",
                       "The benchmark HO contract. Bloomberg HO maps to this."),
            "022653": ("Asian-style options", "Flat price - options",
                       "Options on HO flat price."),
            "022654": ("European-style options", "Flat price - options",
                       "Options on HO flat price."),
            "022A05": ("Cash-settled futures", "Flat price - financial",
                       "Outright ULSD flat price for USGC delivery."),
            "02265V": ("Cash-settled swap", "Flat price - financial",
                       "Gasoil (ICE) swap. Added per Marouen."),
            "022A24": ("Swap", "Flat price - calendar swap",
                       "Gasoil (ICE) mini cal swap. Added per Marouen."),
            "022A46": ("Asian-style options", "Flat price - options",
                       "Gasoil avg price options. Added per Marouen."),
            "02265J": ("Swap", "Flat price - financial",
                       "Singapore gasoil swap. Added per Marouen."),
            "022A22": ("Swap", "Flat price - financial",
                       "Singapore gasoil balmo swap. Added per Marouen."),
        },
    },
    "Gasoline (RBOB)": {
        "bloomberg_ticker": "XB",
        "codes": {
            "111659": ("Physically-delivered futures", "Flat price - benchmark",
                       "The benchmark RBOB contract. Bloomberg XB maps to this."),
            "11165J": ("Cash-settled futures", "Flat price - financial",
                       "RBOB financial settlement."),
            "111416": ("Cash-settled futures", "Flat price - financial",
                       "ICE Energy Div RBOB flat price."),
            "11165K": ("Calendar swap", "Flat price - calendar swap",
                       "RBOB calendar swap. Added per Marouen."),
            "111A11": ("Cash-settled swap", "Flat price - financial",
                       "Singapore MOGAS 92. Added per Marouen."),
            "111A47": ("Futures", "Flat price - financial",
                       "Mini Eurobob gasoline. Added per Marouen."),
        },
    },
    "Gasoil": {
        "bloomberg_ticker": "QS",
        "codes": {
            "02265V": ("Cash-settled swap", "Flat price - financial",
                       "NYMEX-listed swap referencing ICE Gasoil flat price."),
            "022A46": ("Asian-style options", "Flat price - options",
                       "Options on Gasoil flat price."),
            "022A24": ("Swap", "Flat price - calendar swap",
                       "Gasoil (ICE) mini cal swap. Added per Marouen."),
            "02265J": ("Swap", "Flat price - financial",
                       "Singapore gasoil swap. Added per Marouen."),
            "022A22": ("Swap", "Flat price - financial",
                       "Singapore gasoil balmo swap. Added per Marouen."),
        },
    },
}


## Configuration  -  ICE Europe Flat Price Contracts

These have no `cftc_contract_market_code`  -  they come from `downloads/ice/ice_cot.csv`.

In [3]:
ICE_FLAT_PRICE = [
    {
        "future": "Brent Crude Oil",
        "bloomberg_ticker": "CO",
        "market_and_exchange_names": "ICE Brent Crude Futures and Options - ICE Futures Europe",
        "contract_type": "Combined futures + options",
        "category": "Flat price  -  benchmark",
        "rationale": "Physically-deliverable benchmark. Largest Brent OI globally. Bloomberg CO maps to this.",
    },
    {
        "future": "Gasoil",
        "bloomberg_ticker": "QS",
        "market_and_exchange_names": "ICE Gasoil Futures and Options - ICE Futures Europe",
        "contract_type": "Combined futures + options",
        "category": "Flat price  -  benchmark",
        "rationale": "The benchmark Gasoil contract. Bloomberg QS maps to this.",
    },
]

## Configuration  -  Excluded Codes Per Future

Each entry: `code: (category, rationale)`

In [4]:
EXCLUDED = {
    "WTI Crude Oil": {
        "bloomberg_ticker": "CL",
        "codes": {
            # Brent flat price (belongs to CO)
            "06765J": ("Brent flat price", "Brent flat price - mapped to CO, not CL."),
            "06765N": ("Brent flat price", "Brent calendar swap - mapped to CO, not CL."),
            "06765T": ("Brent flat price", "Brent flat price - mapped to CO, not CL."),
            "06765X": ("Brent flat price", "Brent options - mapped to CO, not CL."),
            "06765Y": ("Brent flat price", "Brent options - mapped to CO, not CL."),
            # Other crude grades
            "067DU1": ("Other crude grade", "Oman crude - different grade and delivery."),
            "067A49": ("Other crude grade", "Western Canadian Select - different grade."),
            "067A75": ("Other crude grade", "WCS index - different grade."),
            # Inter-commodity spreads
            "06765L": ("Inter-commodity spread", "WTI vs Brent spread."),
            "06765Z": ("Inter-commodity spread", "WTI vs Brent spread option."),
            "06765O": ("Inter-commodity spread", "Brent vs Dubai spread."),
            "06765M": ("Inter-commodity spread", "Brent structure spread (dated vs futures)."),
            "06741R": ("Inter-commodity spread", "WTI vs Brent spread."),
            # Location / quality diffs
            "06742R": ("Location / quality diff", "Mars vs WTI quality diff."),
            "06742U": ("Location / quality diff", "WTI Midland vs Cushing basis."),
            "06743D": ("Location / quality diff", "WTI Houston vs Cushing basis."),
            "067A71": ("Location / quality diff", "WTI Midland vs Cushing basis."),
            "0676A5": ("Location / quality diff", "WTI Houston vs Cushing basis."),
            "0676A6": ("Location / quality diff", "WTI Houston vs Cushing balmo basis."),
            "06739B": ("Location / quality diff", "WCS vs WTI diff."),
            "06739C": ("Location / quality diff", "WCS vs WTI Houston diff."),
            "06742G": ("Location / quality diff", "Canadian TMX pipeline basis."),
            "06742T": ("Location / quality diff", "Canadian TMX pipeline basis."),
            "06743A": ("Location / quality diff", "Condensate vs TMX diff."),
            # Calendar spread options
            "067657": ("Calendar spread", "Time spread options."),
            "067A28": ("Calendar spread", "Financial calendar spread options."),
            # Crack spreads
            "022A09": ("Crack spread", "ULSD vs crude crack."),
            "86565A": ("Crack spread", "Fuel oil vs crude crack."),
            "86565C": ("Crack spread", "Fuel oil vs crude crack."),
            "86565D": ("Crack spread", "Fuel oil vs crude crack."),
            "86565G": ("Crack spread", "Fuel oil vs crude crack."),
            "86565N": ("Crack spread", "Fuel oil vs Brent crack."),
            "865392": ("Crack spread", "Marine fuel vs Brent crack."),
            "86665A": ("Crack spread", "Naphtha vs crude crack."),
            "86665B": ("Crack spread", "Naphtha vs crude crack."),
            "86665E": ("Other product", "Naphtha flat price - not WTI."),
            "86665G": ("Other product", "Naphtha balmo - not WTI."),
            "86765C": ("Crack spread", "Gasoil vs crude crack."),
            "967652": ("Crack spread", "Gasoline vs crude crack."),
            "967654": ("Crack spread", "Eurobob vs crude crack."),
        },
    },
    "Brent Crude Oil": {
        "bloomberg_ticker": "CO",
        "codes": {
            "06765L": ("Inter-commodity spread", "Inter-commodity spread."),
            "06765Z": ("Inter-commodity spread", "Inter-commodity spread option."),
            "06765O": ("Location spread", "Location spread."),
            "06765M": ("Structure spread", "Structure spread (dated vs futures)."),
            "111415": ("Crack spread", "Crack spread."),
            "111A41": ("Crack spread", "Crack spread."),
            "86565N": ("Crack spread", "Crack spread."),
            "865392": ("Crack spread", "Crack spread."),
            "02141R": ("Crack spread", "Crack spread."),
        },
    },
    "Heating Oil / ULSD": {
        "bloomberg_ticker": "HO",
        "codes": {
            # Crack spreads
            "86765C": ("Crack spread", "Gasoil vs crude spread."),
            "022A09": ("Crack spread", "USGC ULSD vs crude crack."),
            # Location / quality spreads
            "022A26": ("Location / quality spread", "Rotterdam barge vs ICE Gasoil."),
            "02265U": ("Location / quality spread", "HO vs Rotterdam Gasoil arb."),
            "02265T": ("Location spread", "Singapore vs ICE Gasoil location spread."),
            "022A60": ("Location / quality spread", "Group 3 ULSD vs NY Harbor basis."),
            "022A66": ("Location / quality spread", "ULSD vs legacy heating oil spec diff."),
            # Jet / inter-commodity spreads
            "86465D": ("Inter-commodity spread", "Jet fuel vs Gasoil spread."),
            "86465A": ("Inter-commodity spread", "Jet vs HO spread."),
            "86465C": ("Inter-commodity spread", "Jet kerosene vs Gasoil spread."),
            "86465K": ("Calendar spread", "Jet calendar balmo."),
            # Calendar spreads
            "022A18": ("Calendar spread", "Calendar roll spread."),
            "022A62": ("Calendar spread", "Balance-of-month calendar spread."),
            "022A13": ("Calendar spread", "Gulf Coast vs NY calendar + location spread."),
            # Biodiesel
            "869652": ("Biodiesel spread", "Biodiesel vs Gasoil spread."),
            "869654": ("Biodiesel spread", "Biodiesel vs Gasoil spread."),
        },
    },
    "Gasoline (RBOB)": {
        "bloomberg_ticker": "XB",
        "codes": {
            # Calendar spreads
            "11165L": ("Calendar spread", "Up-down calendar spread."),
            # Regional grades / location basis
            "111A08": ("Location / quality spread", "Group 3 sub-octane vs RBOB basis."),
            "111A31": ("Location / quality spread", "Gulf Coast unleaded vs RBOB basis."),
            "111A34": ("Location / quality spread", "Gulf Coast CBOB vs RBOB basis."),
            # Crack spreads
            "111415": ("Crack spread", "RBOB vs Brent crack."),
            "111A41": ("Crack spread", "RBOB vs Brent crack."),
            "967652": ("Crack spread", "RBOB vs WTI crack."),
            "967654": ("Crack spread", "Eurobob vs crude crack."),
        },
    },
    "Gasoil": {
        "bloomberg_ticker": "QS",
        "codes": {
            # Location spreads
            "022A26": ("Location / quality spread", "Rotterdam barge vs ICE Gasoil location/quality spread."),
            "02265U": ("Location spread", "HO vs Rotterdam Gasoil arb."),
            "02265T": ("Location spread", "Singapore vs ICE Gasoil location spread."),
            # Crack spreads
            "86765C": ("Crack spread", "Gasoil vs crude spread."),
            # Jet / biodiesel
            "86465C": ("Inter-commodity spread", "Jet kerosene vs Gasoil spread."),
            "86465D": ("Inter-commodity spread", "Jet fuel vs Gasoil spread."),
            "869652": ("Biodiesel spread", "Biodiesel vs Gasoil spread."),
            "869654": ("Biodiesel spread", "Biodiesel vs Gasoil spread."),
        },
    },
}

## Load Source Data

We need the CFTC data to look up `market_and_exchange_names` and `commodity_name`
for each contract code.

In [5]:
df_cftc = pd.read_csv("../downloads/cftc/disaggregated_combined.csv", low_memory=False)
df_cftc["open_interest_all"] = pd.to_numeric(df_cftc["open_interest_all"], errors="coerce")
df_cftc["report_date_as_yyyy_mm_dd"] = pd.to_datetime(df_cftc["report_date_as_yyyy_mm_dd"])

# Build a lookup: code -> (first market_and_exchange_names, first commodity_name)
code_lookup = (
    df_cftc.groupby("cftc_contract_market_code")
    .agg(
        market_and_exchange_names=("market_and_exchange_names", "first"),
        commodity_name=("commodity_name", "first"),
    )
    .to_dict(orient="index")
)

# Build a lookup: code -> (last report date, last report OI)
latest_per_code = (
    df_cftc.sort_values("report_date_as_yyyy_mm_dd")
    .groupby("cftc_contract_market_code")
    .agg(
        last_report_date=("report_date_as_yyyy_mm_dd", "max"),
        last_report_oi=("open_interest_all", "last"),
    )
    .to_dict(orient="index")
)

# Load ICE data for latest date/OI
df_ice = pd.read_csv("../downloads/ice/ice_cot.csv", low_memory=False)
df_ice["Open_Interest_All"] = pd.to_numeric(df_ice["Open_Interest_All"], errors="coerce")
df_ice["report_date"] = pd.to_datetime(df_ice["As_of_Date_Form_MM/DD/YYYY"])

ice_latest = {}
for name_filter in ["Brent Crude", "Gasoil"]:
    subset = df_ice[df_ice["Market_and_Exchange_Names"].str.contains(name_filter, case=False, na=False)]
    if len(subset) > 0:
        last_row = subset.sort_values("report_date").iloc[-1]
        ice_latest[name_filter] = {
            "last_report_date": last_row["report_date"],
            "last_report_oi": last_row["Open_Interest_All"],
        }

print(f"Loaded {len(code_lookup):,} unique CFTC contract codes")
print(f"ICE latest: {list(ice_latest.keys())}")

Loaded 673 unique CFTC contract codes
ICE latest: ['Brent Crude', 'Gasoil']


## Build the CSV Rows

In [6]:
rows = []

# Map ICE contract names to their latest data
ICE_LATEST_MAP = {
    "ICE Brent Crude Futures and Options - ICE Futures Europe": "Brent Crude",
    "ICE Gasoil Futures and Options - ICE Futures Europe": "Gasoil",
}

# Latest CFTC report date across all contracts
latest_cftc_date = df_cftc["report_date_as_yyyy_mm_dd"].max()
# Latest ICE report date across all contracts
latest_ice_date = df_ice["report_date"].max()

print(f"Latest CFTC report date: {latest_cftc_date.date()}")
print(f"Latest ICE report date: {latest_ice_date.date()}")

# --- ICE Europe flat price contracts (no CFTC code) ---
for ice in ICE_FLAT_PRICE:
    ice_key = ICE_LATEST_MAP.get(ice["market_and_exchange_names"], "")
    ice_data = ice_latest.get(ice_key, {})
    last_date = ice_data.get("last_report_date", pd.NaT)
    active = "Yes" if (pd.notna(last_date) and last_date >= latest_ice_date) else "No"
    rows.append({
        "future": ice["future"],
        "bloomberg_ticker": ice["bloomberg_ticker"],
        "cftc_contract_market_code": "",
        "market_and_exchange_names": ice["market_and_exchange_names"],
        "commodity_name": "",
        "source": "ICE",
        "contract_type": ice["contract_type"],
        "flat_price_selected": "Yes",
        "category": ice["category"],
        "rationale": ice["rationale"],
        "last_report_date": str(last_date.date()) if pd.notna(last_date) else "",
        "last_report_oi": ice_data.get("last_report_oi", ""),
        "currently_active": active,
    })

# --- CFTC flat price contracts ---
for future_name, cfg in FLAT_PRICE.items():
    ticker = cfg["bloomberg_ticker"]
    for code, (contract_type, category, rationale) in cfg["codes"].items():
        info = code_lookup.get(code, {})
        latest = latest_per_code.get(code, {})
        last_date = latest.get("last_report_date", "")
        active = "Yes" if (hasattr(last_date, "date") and last_date >= latest_cftc_date) else "No"
        rows.append({
            "future": future_name,
            "bloomberg_ticker": ticker,
            "cftc_contract_market_code": code,
            "market_and_exchange_names": info.get("market_and_exchange_names", ""),
            "commodity_name": info.get("commodity_name", ""),
            "source": "CFTC",
            "contract_type": contract_type,
            "flat_price_selected": "Yes",
            "category": category,
            "rationale": rationale,
            "last_report_date": str(last_date.date()) if hasattr(last_date, "date") else "",
            "last_report_oi": latest.get("last_report_oi", ""),
            "currently_active": active,
        })

# --- CFTC excluded contracts ---
for future_name, cfg in EXCLUDED.items():
    ticker = cfg["bloomberg_ticker"]
    for code, (category, rationale) in cfg["codes"].items():
        info = code_lookup.get(code, {})
        latest = latest_per_code.get(code, {})
        last_date = latest.get("last_report_date", "")
        active = "Yes" if (hasattr(last_date, "date") and last_date >= latest_cftc_date) else "No"
        # Infer contract_type from the contract name
        name = info.get("market_and_exchange_names", "")
        if "OPTION" in name.upper():
            contract_type = "Spread option"
        elif "SWAP" in name.upper() or "SWP" in name.upper() or "SPR" in name.upper():
            contract_type = "Swap"
        elif "INDEX" in name.upper():
            contract_type = "Index swap"
        else:
            contract_type = "Futures"
        rows.append({
            "future": future_name,
            "bloomberg_ticker": ticker,
            "cftc_contract_market_code": code,
            "market_and_exchange_names": name,
            "commodity_name": info.get("commodity_name", ""),
            "source": "CFTC",
            "contract_type": contract_type,
            "flat_price_selected": "No",
            "category": category,
            "rationale": rationale,
            "last_report_date": str(last_date.date()) if hasattr(last_date, "date") else "",
            "last_report_oi": latest.get("last_report_oi", ""),
            "currently_active": active,
        })

result = pd.DataFrame(rows)

# Sort: by future, then selected first, then by code
future_order = ["WTI Crude Oil", "Brent Crude Oil", "Heating Oil / ULSD", "Gasoline (RBOB)", "Gasoil"]
result["_future_rank"] = result["future"].map({f: i for i, f in enumerate(future_order)})
result["_selected_rank"] = result["flat_price_selected"].map({"Yes": 0, "No": 1})
result = result.sort_values(["_future_rank", "_selected_rank", "cftc_contract_market_code"]).drop(
    columns=["_future_rank", "_selected_rank"]
)

print(f"\nTotal rows: {len(result)}")
print(f"Selected: {(result['flat_price_selected'] == 'Yes').sum()}")
print(f"Excluded: {(result['flat_price_selected'] == 'No').sum()}")
print(f"Currently active: {(result['currently_active'] == 'Yes').sum()}")
print(f"Inactive: {(result['currently_active'] == 'No').sum()}")
result.head(10)

Latest CFTC report date: 2026-03-31
Latest ICE report date: 2026-03-24

Total rows: 118
Selected: 37
Excluded: 81
Currently active: 31
Inactive: 87


,future,bloomberg_ticker,cftc_contract_market_code,market_and_exchange_names,commodity_name,source,contract_type,flat_price_selected,category,rationale,last_report_date,last_report_oi,currently_active
3,WTI Crude Oil,CL,067411,"CRUDE OIL, LIGHT SWEET - ICE EUROPE",CRUDE OIL,CFTC,Futures (WTI-referenced),Yes,Flat price - benchmark,"ICE-listed WTI lookalike, reported by CFTC. Ag...",2026-03-31,1184319.0,Yes
9,WTI Crude Oil,CL,06741Q,WTI CRUDE OIL 1ST LINE - ICE FUTURES ENERGY DIV,CRUDE OIL,CFTC,Cash-settled futures,Yes,Flat price - financial,ICE Energy Div WTI flat price.,2026-03-31,38890.0,Yes
2,WTI Crude Oil,CL,067651,"CRUDE OIL, LIGHT SWEET - NEW YORK MERCANTILE E...",CRUDE OIL,CFTC,Physically-delivered futures,Yes,Flat price - benchmark,The benchmark WTI contract. Bloomberg CL maps ...,2026-03-31,3039583.0,Yes
6,WTI Crude Oil,CL,067655,"E-MINI CRUDE OIL, LIGHT SWEET - NEW YORK MERCA...",CRUDE OIL,CFTC,E-mini futures,Yes,Flat price - financial,Smaller-sized WTI flat price contract.,2007-01-16,24796.0,No
10,WTI Crude Oil,CL,06765A,WTI CRUDE OIL CALENDAR SWAP - NEW YORK MERCANT...,CRUDE OIL,CFTC,Calendar swap,Yes,Flat price - calendar swap,WTI calendar swap - settles against average pr...,2026-03-31,235004.0,Yes
8,WTI Crude Oil,CL,06765B,EUR STYLE CRUDE OIL OPTIONS - NEW YORK MERCANT...,CRUDE OIL,CFTC,European-style options,Yes,Flat price - options,Options on WTI flat price.,2022-08-16,40152.0,No
7,WTI Crude Oil,CL,06765C,CRUDE OIL AVG PRICE OPTIONS - NEW YORK MERCANT...,CRUDE OIL,CFTC,Asian-style options,Yes,Flat price - options,Options on WTI flat price.,2026-03-31,173578.0,Yes
11,WTI Crude Oil,CL,06765G,DUBAI CRUDE OIL CALENDAR SWAP - NEW YORK MERCA...,CRUDE OIL,CFTC,Calendar swap,Yes,Flat price - calendar swap,Dubai crude calendar swap. Added per Marouen.,2012-10-30,30649.0,No
4,WTI Crude Oil,CL,06765I,WTI CRUDE OIL FINANCIAL - NEW YORK MERCANTILE ...,CRUDE OIL,CFTC,Cash-settled futures,Yes,Flat price - financial,WTI financial settlement - flat price exposure...,2013-09-17,48209.0,No
5,WTI Crude Oil,CL,067A55,Micro WTI CRUDE OIL FINANCIAL - NEW YORK MERCA...,CRUDE OIL,CFTC,Cash-settled micro futures,Yes,Flat price - financial,Micro-sized WTI flat price contract.,2026-03-31,76474.0,Yes


## Write to CSV

In [7]:
output_path = Path("../docs/cot_contract_mapping_summary.csv")
result.to_csv(output_path, index=False, quoting=csv.QUOTE_NONNUMERIC, encoding="utf-8-sig")
print(f"Written to {output_path.resolve()}")
print(f"File size: {output_path.stat().st_size:,} bytes")

Written to /Users/oualid/Documents/Projects/omroot_repos/cot-ingest/docs/cot_contract_mapping_summary.csv
File size: 25,591 bytes


## Summary by Future

In [8]:
summary = (
    result.groupby(["future", "bloomberg_ticker", "flat_price_selected"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={"Yes": "selected", "No": "excluded"})
)
summary["total"] = summary["selected"] + summary["excluded"]
summary

,flat_price_selected,excluded,selected,total
future,bloomberg_ticker,,,
Brent Crude Oil,CO,9,6,15
Gasoil,QS,8,6,14
Gasoline (RBOB),XB,8,6,14
Heating Oil / ULSD,HO,16,9,25
WTI Crude Oil,CL,40,10,50


## Excluded Categories Breakdown

In [9]:
excluded = result[result["flat_price_selected"] == "No"]
excluded.groupby(["future", "category"]).size().unstack(fill_value=0)

category,Biodiesel spread,Brent flat price,Calendar spread,Crack spread,Inter-commodity spread,Location / quality diff,Location / quality spread,Location spread,Other crude grade,Other product,Structure spread
future,,,,,,,,,,,
Brent Crude Oil,0,0,0,5,2,0,0,1,0,0,1
Gasoil,2,0,0,1,2,0,1,2,0,0,0
Gasoline (RBOB),0,0,1,4,0,0,3,0,0,0,0
Heating Oil / ULSD,2,0,4,2,3,0,4,1,0,0,0
WTI Crude Oil,0,5,2,12,5,11,0,0,3,2,0
